In [ ]:
!pip install --upgrade --force-reinstall numpy
!pip install --upgrade --force-reinstall opencv-python
pip install tensorflow


In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers, models


DATASET_PATH = "C:/Users/antho/Downloads/Seizure_detection_10sec/Seizure_detection_10sec"     # chemin d'accès au dataset 
FPS = 30                # fréquence (frame par sec)
DURATION = 10 

In [ ]:
# Vérification des sous-dossiers


for label in ['0', '1']:
    path = os.path.join(DATASET_PATH, label)
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"✅ Dossier {label} trouvé : {len(files)} fichiers détectés.")
    else:
        print(f"❌ Erreur : Dossier {label} introuvable dans {DATASET_PATH}")

✅ Dossier 0 trouvé : 10101 fichiers détectés.
✅ Dossier 1 trouvé : 2952 fichiers détectés.


Optical flow & extraction de features

In [ ]:
def extract_video_features(video_path):
    cap = cv2.VideoCapture(video_path)        # ouverture de la vidéo 
    features = []                            # liste pour stocker les caractéristiques de chaque frame
    
    ret, frame1 = cap.read()                   
    if not ret: return None
    prev_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    
    while True:                      #
        ret, frame2 = cap.read()    # lecture de la frame suivante
        if not ret: break
        
        gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        #    calcul de Flux optique 
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)     # paramètres pour le calcul du flux optique
        
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])         
        
        # Extraction des caractéristiques par frame (FlowFeat)
        frame_feat = [
            np.mean(mag), # Intensité moyenne
            np.var(mag),  # Variance (agitation)
            np.max(mag),  # Pic de mouvement
            np.std(mag)   # Écart-type
        ]
        features.append(frame_feat)
        prev_gray = gray
        
    cap.release()
    return np.array(features)

 Affichage des signaux 

In [ ]:
# Sélection d'une vidéo de crise pour la démo
sample_video = os.path.join(DATASET_PATH, "1", os.listdir(os.path.join(DATASET_PATH, "1"))[0])    # ( [0] :   premiere vidéo  du dossier 1)
sample_feats = extract_video_features(sample_video)

#  SIGNAL TEMPOREL 
plt.figure(figsize=(12, 4))
plt.plot(sample_feats[:, 0], color='red', label='Magnitude Moyenne')
plt.title("Étape 2 : Évolution du mouvement (Vidéo de Crise)")
plt.xlabel("Frames (Temps)")
plt.ylabel("Intensité du Flux Optique")
plt.legend()
plt.grid(True)
plt.show()

# ANALYSE FRÉQUENTIELLE (FFT) ---
signal = sample_feats[:, 0] - np.mean(sample_feats[:, 0]) # On retire la composante continue
n = len(signal)
yf = fft(signal)
xf = fftfreq(n, 1/FPS)[:n//2]

plt.figure(figsize=(12, 4))
plt.plot(xf, 2.0/n * np.abs(yf[0:n//2]), color='blue')
plt.title("Étape 3 : Signature Fréquentielle (FFT) de la Crise")
plt.xlabel("Fréquence (Hz)")
plt.ylabel("Amplitude")
plt.xlim(0, 10) # On zoome sur les basses fréquences (0-10Hz)
plt.grid(True)
plt.show()

Préparation des données

In [ ]:
import random
from tqdm import tqdm

X = []      
Y = []      

# limitation à  1000 videos PAr classe
LIMIT_PER_CLASS = 1000

for label in ['0', '1']:
    folder = os.path.join(DATASET_PATH, label)
    # Lister toutes les vidéos valides
    all_videos = [f for f in os.listdir(folder) if f.endswith(('.mp4', '.avi', '.mkv'))]
    
    # mélange des vidéos pour éviter les biais de sélection
    random.seed(42)
    random.shuffle(all_videos)
    
    # Sélectionner les 1000 premières
    selected_videos = all_videos[:LIMIT_PER_CLASS]
    
    print(f"\nTraitement du dossier {label} : Extraction de {len(selected_videos)} vidéos...")
    
    # Barre de progression tqdm
    for vid_name in tqdm(selected_videos):
        path = os.path.join(folder, vid_name)
        f = extract_video_features(path)
        
        # Vérification de la longueur (30 FPS * 10 sec = 300 frames, on en prend 290 par sécurité)
        if f is not None and len(f) >= 290:
            X.append(f[:290]) 
            Y.append(int(label))

X = np.array(X)
Y = np.array(Y)

# Division Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print(f"\n Terminé !")
print(f"Total vidéos chargées : {len(X)}")
print(f"Taille de X_train : {X_train.shape}")


Traitement du dossier 0 : Extraction de 1000 vidéos...


100%|██████████| 1000/1000 [46:44<00:00,  2.80s/it] 



Traitement du dossier 1 : Extraction de 1000 vidéos...


100%|██████████| 1000/1000 [1:03:44<00:00,  3.82s/it]



✅ Terminé !
Total vidéos chargées : 2000
Taille de X_train : (1600, 290, 4)


Sauvegarde

In [17]:
# Sauvegarde locale
np.save('X_balanced.npy', X)
np.save('Y_balanced.npy', Y)
print("Données sauvegardées sous 'X_balanced.npy' et 'Y_balanced.npy'")

Données sauvegardées sous 'X_balanced.npy' et 'Y_balanced.npy'


In [ ]:
# On aplatit les données pour la régression (Moyenne globale par vidéo)
X_train_flat = np.mean(X_train, axis=1)
X_test_flat = np.mean(X_test, axis=1)

log_reg = LogisticRegression()
log_reg.fit(X_train_flat, y_train)

y_pred_lr = log_reg.predict(X_test_flat)
print("Résultats Régression Logistique :")
print(classification_report(y_test, y_pred_lr))

# calcul de précision, rappel, f1-score
print("Précision:", np.round(np.mean(y_pred_lr == y_test), 4))
print("Rappel:", np.round(np.sum((y_pred_lr == 1) & (y_test == 1)) / np.sum(y_test == 1), 4)*100, "%")
print("F1-Score:", np.round(2 * (np.mean(y_pred_lr == y_test) * (np.sum((y_pred_lr == 1) & (y_test == 1)) / np.sum(y_test == 1))) / (np.mean(y_pred_lr == y_test) + (np.sum((y_pred_lr == 1) & (y_test == 1)) / np.sum(y_test == 1))), 4))



# ETAPE CNN

In [ ]:
model = models.Sequential([
    layers.Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2)

# matrice de confusion
y_pred_cnn = (model.predict(X_test) > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_cnn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Crise'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matrice de Confusion Finale (CNN 1D)")
plt.show()


#calcul de la précision, rappel et F1-score
print("Résultats CNN 1D :")
print(classification_report(y_test, y_pred_cnn))

print("Précision : ", classification_report(y_test, y_pred_cnn, output_dict=True)['1']['precision'])
print("Rappel : ", classification_report(y_test, y_pred_cnn, output_dict=True)['1']['recall'])
print("F1-score : ", classification_report(y_test, y_pred_cnn, output_dict=True)['1']['f1-score'])




In [ ]:
# mon fit
plt.figure(figsize=(12, 4))

# Graphique de l'Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Précision du Modèle (CNN)')
plt.legend()

# Graphique de la Perte (Loss)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Perte du Modèle (Loss)')
plt.legend()

plt.show()

SVM


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

#  Préparation des données (comme pour la régression, on utilise les moyennes de FlowFeat)
X_train_svm = np.mean(X_train, axis=1)
X_test_svm = np.mean(X_test, axis=1)

# mise à l'échelle
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_svm)
X_test_scaled = scaler.transform(X_test_svm)

# . Création et entraînement du SVM
# On utilise C=1.0 et un noyau 'rbf' par défaut
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 4. Prédictions
y_pred_svm = svm_model.predict(X_test_scaled)

# 5. Affichage des résultats
print("--- RÉSULTATS SVM ---")
print(classification_report(y_test, y_pred_svm))

# Matrice de confusion pour le SVM
plt.figure(figsize=(6, 5))
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=['Normal', 'Crise'])
disp_svm.plot(cmap=plt.cm.Greens)
plt.title("Matrice de Confusion : SVM")
plt.show()


# COurbe de séparation pour le SVM
# Obtenir les probabilités de classe pour le SVM
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]
plt.figure(figsize=(8, 5))
plt.hist(y_prob_svm[y_test == 0], bins=20, alpha=0.5, label='Normal', color='green')
plt.hist(y_prob_svm[y_test == 1], bins=20, alpha=0.5, label='Crise', color='red')
plt.title("Distribution des Probabilités de Classe (SVM)")
plt.xlabel("Probabilité de Crise")
plt.ylabel("Nombre de Vidéos")
plt.legend()
plt.show()

In [ ]:
# séparation des classes pour le SVM
plt.figure(figsize=(8, 5))
plt.scatter(X_test_scaled[y_test == 0][:, 0], X_test_scaled[y_test == 0][:, 1], label='Normal', alpha=0.5, color='green')
plt.scatter(X_test_scaled[y_test == 1][:, 0], X_test_scaled[y_test == 1][:, 1], label='Crise', alpha=0.5, color='red')
plt.title("Séparation des Classes (SVM)")
plt.xlabel("Caractéristique 1 (Moyenne de FlowFeat)")
plt.ylabel("Caractéristique 2 (Variance de FlowFeat)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def generate_motion_heatmap(video_path, save_path=None):
    cap = cv2.VideoCapture(video_path)
    ret, frame1 = cap.read()
    if not ret:
        return None
    
    
    prev_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    # Création d'une matrice vide pour accumuler les amplitudes de mouvement
    motion_accumulator = np.zeros_like(prev_gray, dtype=np.float32)
    
    count = 0
    while True:
        ret, frame2 = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        

        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Calcul de la magnitude (intensité du déplacement de chaque pixel)
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        
        # Accumulation du mouvement sur chaque pixel
        motion_accumulator += mag
        
        prev_gray = gray
        count += 1
    
    cap.release()
    
    # Normalisation pour l'affichage (0-255)
    heatmap = cv2.normalize(motion_accumulator, None, 0, 255, cv2.NORM_MINMAX)
    heatmap = np.uint8(heatmap)
    
    # Application d'une palette de couleur  (Bleu vers Rouge)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    return heatmap_color


# 1. Vidéo de Crise (Label 1)
video_1 = os.path.join(DATASET_PATH, "1", os.listdir(os.path.join(DATASET_PATH, "1"))[1])
heatmap_crise = generate_motion_heatmap(video_1)

# 2. Vidéo Normale (Label 0)
video_0 = os.path.join(DATASET_PATH, "0", os.listdir(os.path.join(DATASET_PATH, "0"))[1])
heatmap_normal = generate_motion_heatmap(video_0)

# Affichage côte à côte
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

ax1.imshow(cv2.cvtColor(heatmap_crise, cv2.COLOR_BGR2RGB))
ax1.set_title("Heatmap : Activité de Crise (Intense)")
ax1.axis('off')

ax2.imshow(cv2.cvtColor(heatmap_normal, cv2.COLOR_BGR2RGB))
ax2.set_title("Heatmap : Activité Normale (Calme)")
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def generate_motion_heatmap(video_path, save_path=None):
    cap = cv2.VideoCapture(video_path)
    ret, frame1 = cap.read()
    if not ret:
        return None
    
    
    prev_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
   
    motion_accumulator = np.zeros_like(prev_gray, dtype=np.float32)
    
    count = 0
    while True:
        ret, frame2 = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        

        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        

        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        

        motion_accumulator += mag
        
        prev_gray = gray
        count += 1
    
    cap.release()
    

    heatmap = cv2.normalize(motion_accumulator, None, 0, 255, cv2.NORM_MINMAX)
    heatmap = np.uint8(heatmap)
    

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    return heatmap_color



# 1. Vidéo de Crise (Label 1)
video_1 = os.path.join(DATASET_PATH, "1", os.listdir(os.path.join(DATASET_PATH, "1"))[5])    # vidé
heatmap_crise = generate_motion_heatmap(video_1)

# 2. Vidéo Normale (Label 0)
video_0 = os.path.join(DATASET_PATH, "0", os.listdir(os.path.join(DATASET_PATH, "0"))[5])
heatmap_normal = generate_motion_heatmap(video_0)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

ax1.imshow(cv2.cvtColor(heatmap_crise, cv2.COLOR_BGR2RGB))
ax1.set_title("Heatmap : Activité de Crise (Intense/Chaotique)")
ax1.axis('off')

ax2.imshow(cv2.cvtColor(heatmap_normal, cv2.COLOR_BGR2RGB))
ax2.set_title("Heatmap : Activité Normale (Localisée/Calme)")
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#heatmap pour des videos de crise et normales pour montrer la différence d'intensité et de localisation du mouvement

for label in ['0', '1']:
    folder = os.path.join(DATASET_PATH, label)
    video_path = os.path.join(folder, os.listdir(folder)[0])  # Prendre la première vidéo du dossier
    heatmap = generate_motion_heatmap(video_path)
    
    plt.figure(figsize=(6, 5))
    plt.imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
    plt.title(f"Heatmap de Mouvement - Label {label} ({'Crise' if label == '1' else 'Normal'})")
    plt.axis('off')
    plt.show()
    

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os



# Fonction pour calculer et afficher le champ de vecteurs de flux optique (quiver) pour une vidéo donnée


def plot_quiver_flow(video_path, frame_index=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Erreur : impossible d'ouvrir la vidéo :", video_path)
        return
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    
    ret, frame1 = cap.read()
    if not ret:
        print(f"Erreur : impossible de lire la frame {frame_index}.")
        return
    
    ret, frame2 = cap.read()
    if not ret:
        print(f"Erreur : impossible de lire la frame {frame_index+1}.")
        return
    
    prev_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                        0.5, 3, 15, 3, 5, 1.2, 0)

    print("Flow shape:", flow.shape)

    step = 10
    y, x = np.mgrid[step/2:flow.shape[0]:step,
                    step/2:flow.shape[1]:step].reshape(2,-1).astype(int)
    fx, fy = flow[y, x].T

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
    plt.quiver(x, y, fx, -fy, color='lime', scale=20, width=0.003)
    plt.title(f"Champ de vecteurs (u, v) - Frame {frame_index}")
    plt.axis('off')
    plt.show()

# Exemple d'utilisation
DATASET_PATH = "C:/Users/antho/Downloads/Seizure_detection_10sec/Seizure_detection_10sec"
video_sample = os.path.join(DATASET_PATH, "1", os.listdir(os.path.join(DATASET_PATH, "1"))[15]) 
plot_quiver_flow(video_sample, frame_index=30)




In [ ]:
# score AUC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

def plot_courbe_roc(y_true, y_scores):
    from sklearn.metrics import roc_curve, auc
    
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--')
    plt.title('Courbe ROC')
    plt.xlabel('Taux de Faux Positifs (FPR)')
    plt.ylabel('Taux de Vrais Positifs (TPR)')
    plt.legend(loc='lower right')
    plt.grid()
    plt.show()